In [ ]:
# case study -1 restaurant reviews predictions
# many to one
# LSTM/GRU

Hi


In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import pandas as pd

In [3]:
df=pd.read_csv('r_data.tsv',delimiter='\t')

In [84]:
# Data cleaning

In [4]:
df.isnull().sum()

Review    0
Liked     0
dtype: int64

In [5]:
df.dtypes

Review    object
Liked      int64
dtype: object

In [6]:
df['Review']

0                                Wow... Loved this place.
1                                      Crust is not good.
2               Not tasty and the texture was just nasty.
3       Stopped by during the late May bank holiday of...
4       The selection on the menu was great and so wer...
                              ...                        
1386                                                  bad
1387                                             very bad
1388                                             bad food
1389                                            bad space
1390                                          bad service
Name: Review, Length: 1391, dtype: object

In [7]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re

In [8]:
s_list=stopwords.words('english')

In [9]:
s_list.remove('not')

In [10]:
len(s_list)

197

In [11]:
corpus=[]
for i in df['Review']:
    mystr=i.lower()  # lowercase
    newstr=re.sub(r'[^A-Za-z\s]','',mystr) # remove special chracters/digit
    list1=word_tokenize(newstr) # tokenization
    list2=[ i for i in list1 if i not in s_list]  # stopwords removal
    #ps=PorterStemmer()
    #list3=[ ps.stem(i)  for i in list2]  # stemming
    final=' '.join(list2)
    corpus.append(final)

In [12]:
df['Review']=corpus

In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
max_words = 5000
max_len = 100

In [15]:
tk=Tokenizer(num_words=max_words)
tk.fit_on_texts(df['Review'])

In [16]:
import pickle
with open('tokenizer.pkl','wb') as f:
    pickle.dump(tk,f)

In [17]:
X=tk.texts_to_sequences(df['Review'])

In [18]:
X = pad_sequences(X, maxlen=max_len)

In [19]:
X

array([[  0,   0,   0, ..., 328,  97,   4],
       [  0,   0,   0, ..., 475,   2,   5],
       [  0,   0,   0, ...,  27, 329,  68],
       ...,
       [  0,   0,   0, ...,   0,  29,   1],
       [  0,   0,   0, ...,   0,  29, 663],
       [  0,   0,   0, ...,   0,  29,   8]],
      shape=(1391, 100), dtype=int32)

In [20]:
y=df['Liked']

In [21]:
y

0       1
1       0
2       0
3       1
4       1
       ..
1386    0
1387    0
1388    0
1389    0
1390    0
Name: Liked, Length: 1391, dtype: int64

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [23]:
# archtitecture of LSTM
# embedding layer
# LSTM layer
# dropout layer(optional)
# Dense layer

In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [ ]:
# model=Sequential()
# model.add(Embedding(input_dim=max_words, output_dim=128))
# model.add(LSTM(128, return_sequences=False))
# model.add(Dropout(0.5))
# model.add(Dense(1, activation='sigmoid'))

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128))
model.add(Bidirectional(LSTM(128)))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

In [26]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [27]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 35s 377ms/step - accuracy: 0.5917 - loss: 0.6628 - val_accuracy: 0.5484 - val_loss: 0.6771
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 19s 330ms/step - accuracy: 0.7437 - loss: 0.5345 - val_accuracy: 0.8351 - val_loss: 0.4902
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 24s 418ms/step - accuracy: 0.8957 - loss: 0.2926 - val_accuracy: 0.7885 - val_loss: 0.4509
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 11s 320ms/step - accuracy: 0.9667 - loss: 0.1761 - val_accuracy: 0.8351 - val_loss: 0.4630
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 21s 324ms/step - accuracy: 0.9757 - loss: 0.0769 - val_accuracy: 0.8351 - val_loss: 0.4923
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 24s 420ms/step - accuracy: 0.9901 - loss: 0.0422 - val_accuracy: 0.8351 - val_loss: 0.6049
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 12s 335ms/step - accuracy: 0.9955 - loss: 0.0318 - val_accuracy: 0.8351 - val_loss: 0.6951
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 21s 346ms/step - accuracy: 0.9955 - loss: 0.0232 - val_accu

In [60]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.7921 - loss: 1.0466
Test Accuracy: 0.7921146750450134


In [60]:
history = model.fit(
    X,
    y,
    epochs=30,
    batch_size=32)

Epoch 1/30


48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 161ms/step - accuracy: 0.8706 - loss: 0.4761
Epoch 2/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 214ms/step - accuracy: 0.9914 - loss: 0.0500
Epoch 3/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s 146ms/step - accuracy: 0.9974 - loss: 0.0128
Epoch 4/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 8s 158ms/step - accuracy: 1.0000 - loss: 0.0042
Epoch 5/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 13s 216ms/step - accuracy: 0.9993 - loss: 0.0029
Epoch 6/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 20s 197ms/step - accuracy: 1.0000 - loss: 0.0016
Epoch 7/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 167ms/step - accuracy: 1.0000 - loss: 0.0011
Epoch 8/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 212ms/step - accuracy: 1.0000 - loss: 9.0796e-04
Epoch 9/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 8s 162ms/step - accuracy: 1.0000 - loss: 5.8095e-04
Epoch 10/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 11s 168ms/step - accuracy: 1.0000 - loss: 5.1113e-04
Epoch 11/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s 146ms/step - accuracy: 1.0000 - loss: 3.5431e-04
Epoch 12/30
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s

In [28]:
model.save('lstm_reviews.keras')

In [62]:
y_pred=model.predict(X_test)

9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 173ms/step


In [63]:
y_pred1=[  1 if i>0.5 else 0  for i in y_pred]

In [64]:
from sklearn.metrics import confusion_matrix,classification_report

In [65]:
confusion_matrix(y_test,y_pred1)

array([[ 99,  31],
       [ 27, 122]])

In [66]:
print(classification_report(y_test,y_pred1))

              precision    recall  f1-score   support

           0       0.79      0.76      0.77       130
           1       0.80      0.82      0.81       149

    accuracy                           0.79       279
   macro avg       0.79      0.79      0.79       279
weighted avg       0.79      0.79      0.79       279



In [67]:
a='good taste'

In [33]:
def predict_text(text):
    text = text.lower().strip()
    seq = tk.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    pred = model.predict(padded)[0][0]
    print(pred)
    if pred > 0.5:
        return f"Positive ({pred:.2f})"
    else:
        return f"Negative ({pred:.2f})"

In [34]:
predict_text('good')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
0.9976883


'Positive (1.00)'

In [32]:
df

,Review,Liked
0,food terrible service worst,0
1,really enjoyed pizza amazing,1
2,highly recommend place fantastic experience,1
3,amazing burger friendly staff,1
4,service delicious food tasted perfect,1
...,...,...
1495,bad ambiance bad food,0
1496,not worth awful experience,0
1497,staff rude food bad,0
1498,terrible burger rude staff,0


In [35]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# 1. Load Pre-trained Model and Tokenizer
# 'bert-base-uncased' is a popular baseline that ignores letter casing
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Prepare a sample review
review = "This product is absolutely amazing and exceeded my expectations!"

# 3. Tokenize input
# BERT requires specific preprocessing like [CLS] and [SEP] tokens
inputs = tokenizer(review, padding=True, truncation=True, max_length=128, return_tensors="pt")

# 4. Predict
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    # Convert raw logits to probabilities
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(predictions).item()

sentiment = "Positive" if predicted_class == 1 else "Negative"
print(f"Review: {review}")
print(f"Prediction: {sentiment} ({predictions[0][predicted_class]:.2f} confidence)")


ModuleNotFoundError: No module named 'torch'